In [ ]:
from figure_paths import DATA, RESULTS, WORK, FIGURES, setting
from decimal import Decimal
import numpy as np
import xarray as xr
import netCDF4
import matplotlib as mpl
mpl.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image

DPI=800
NC_PATHS={stage: RESULTS / f'DA_TransUNet_stage{stage}_shap_values.nc' for stage in (1,2)}


In [ ]:
EXPECTED_OUTPUTS=["u850","u925","u1000","v850","v925","v1000","omega850","omega925","omega1000"]
EXPECTED_INPUTS={1:["sp","t2m","d2m"],2:["sp","t2m","d2m"]+["pred_"+x for x in EXPECTED_OUTPUTS]}
NEEDED=["shap_values","prediction","expected_value","attribution_sum","efficiency_error","efficiency_tolerance",
    "quality_pass","convergence_pass","convergence_robust_order","block_complete","importance",
    "sample_init_time","sample_target_time","sample_pool_index","x_feature_units","y_feature_units","y_pressure_hpa"]
datasets = {}
for stage,path in NC_PATHS.items():
    with xr.open_dataset(path,engine="netcdf4",decode_timedelta=False,cache=False) as ds:
        assert int(ds.attrs['stage']) == stage
        target="main_model_forecast" if stage==1 else "residual_correction_only"
        assert ds.attrs["target_definition"]==target
        assert ds.attrs["method_id"]=="expected_gradients_stagewise_physical_region_mean_v1"
        assert ds.shap_values.dims==("sample","forecast_time","y_feature","x_feature")
        assert ds.sizes["sample"]==ds.sizes["background"]==100
        np.testing.assert_array_equal(ds.forecast_time.values,[1,2,3,4,5,6])
        np.testing.assert_array_equal(ds.lag_time.values,[-5,-4,-3,-2,-1,0])
        np.testing.assert_array_equal(ds.x_feature.values,EXPECTED_INPUTS[stage])
        np.testing.assert_array_equal(ds.y_feature.values,EXPECTED_OUTPUTS)
        assert ds.y_feature_units.dims==("y_feature",),"Units must be invariant over lead"
        np.testing.assert_array_equal(ds.y_feature_units.values,["m s-1"]*6+["Pa s-1"]*3)
        np.testing.assert_array_equal(ds.y_pressure_hpa.values,[850,925,1000]*3)
        d=ds[NEEDED].load();phi=d.shap_values
        assert phi.shape==(100,6,9,3 if stage==1 else 12)
        assert np.isfinite(phi.values).all()
        assert (phi.values<0).any() and (phi.values>0).any()
        for name in ("quality_pass","convergence_pass","convergence_robust_order","block_complete"):
            assert (d[name].values==1).all(),name
        summed=phi.sum("x_feature")
        np.testing.assert_allclose(summed,d.attribution_sum,atol=2e-13,rtol=2e-13)
        err=summed-(d.prediction-d.expected_value)
        np.testing.assert_allclose(err,d.efficiency_error,atol=2e-13,rtol=2e-13)
        assert (abs(err)<=d.efficiency_tolerance+2e-13).all().item()
        np.testing.assert_allclose(abs(phi).mean("sample"),d.importance,atol=1e-12,rtol=1e-12)
        datasets[stage]=d
for name in ("sample","forecast_time","y_feature","sample_init_time","sample_target_time","sample_pool_index"):
    np.testing.assert_array_equal(datasets[1][name].values,datasets[2][name].values)


In [ ]:
def largest_remainder_strings(percent):
    value=np.asarray(percent,dtype=np.float64)
    assert value.ndim==2 and np.isfinite(value).all() and (value>=0).all()
    np.testing.assert_allclose(value.sum(axis=0),100,atol=2e-12,rtol=0)
    quota=value*100
    integers=np.floor(quota).astype(np.int64)
    for column in range(value.shape[1]):
        remaining=10000-int(integers[:,column].sum())
        assert 0<=remaining<=value.shape[0],remaining
        fractional=quota[:,column]-integers[:,column]
        order=np.lexsort((np.arange(value.shape[0]),-fractional))
        integers[order[:remaining],column]+=1
    assert (integers.sum(axis=0)==10000).all()
    labels=np.array([[f"{n//100}.{n%100:02d}" for n in row] for row in integers])
    for column in range(labels.shape[1]):
        assert sum(Decimal(x) for x in labels[:,column])==Decimal("100.00")
    assert np.max(np.abs(integers/100-value))<=0.010000000001
    return labels,integers

PLOT={}
for stage,d in datasets.items():
    absolute_mean=abs(d.shap_values).mean(("sample","forecast_time")).transpose("x_feature","y_feature")
    total=absolute_mean.sum("x_feature")
    assert np.isfinite(total).all().item() and (total>0).all().item()
    percentage=100*absolute_mean/total
    check=np.empty(percentage.shape)
    for j,output in enumerate(d.y_feature.values):
        raw=np.array([np.abs(d.shap_values.sel(y_feature=output,x_feature=feature).values).mean()
                      for feature in d.x_feature.values])
        check[:,j]=100*raw/raw.sum()
    np.testing.assert_allclose(percentage.values,check,atol=1e-13,rtol=1e-13)
    labels,integers=largest_remainder_strings(percentage.values)
    PLOT[stage]=dict(absolute_mean=absolute_mean,percentage=percentage,display_labels=labels,display_hundredths=integers)


In [ ]:
FIGURE_NAMES={1:"Fig13",2:"FigS20"}
STYLE={1:dict(figsize=(8.8,3.8),left=0.13,bottom=0.39),2:dict(figsize=(8.8,7.8),left=0.22,bottom=0.22)}
mpl.rcParams.update({"font.family":"sans-serif","font.sans-serif":["Arial","DejaVu Sans"],
    "font.size":11,"axes.labelsize":11,"xtick.labelsize":10.5,"ytick.labelsize":11,
    "axes.spines.top":False,"axes.spines.right":False,"axes.linewidth":0.7,
    "pdf.fonttype":42,"svg.fonttype":"none","savefig.facecolor":"white"})
CMAP=LinearSegmentedColormap.from_list("importance_single_hue",["#fffdf8","#fff0cf","#f3cf92","#d9995f"])
OUTPUT_LABELS=["u 850","u 925","u 1000","v 850","v 925","v 1000","ω 850","ω 925","ω 1000"]

def feature_label(name):
    if name.startswith("pred_"):
        return "Pred. "+name[5:].replace("omega","ω").replace("850"," 850").replace("925"," 925").replace("1000"," 1000")
    return name

for stage in (1,2):
    p=PLOT[stage];style=STYLE[stage];z=p["percentage"].values
    fig=plt.figure(figsize=style["figsize"],facecolor="white")
    ax=fig.add_axes([style["left"],style["bottom"],0.97-style["left"],0.94-style["bottom"]])
    vmax=float(max(5,5*np.ceil(z.max()/5)))
    image=ax.imshow(z,cmap=CMAP,vmin=0,vmax=vmax,aspect="auto",interpolation="nearest")
    ax.set_yticks(range(z.shape[0]),[feature_label(str(f)) for f in p["percentage"].x_feature.values])
    ax.set_xticks(range(9),OUTPUT_LABELS,rotation=40,ha="right",rotation_mode="anchor")
    ax.set_xlabel("Output component / pressure (hPa)")
    for row in range(z.shape[0]):
        for column in range(z.shape[1]):
            ax.text(column,row,p["display_labels"][row,column],color="black",ha="center",va="center",fontsize=10.8)
    cb_bottom=0.14 if stage==1 else 0.075
    cb_height=0.04 if stage==1 else 0.023
    cax=fig.add_axes([style["left"],cb_bottom,0.97-style["left"],cb_height])
    step=max(5,5*np.ceil(vmax/6/5));ticks=np.arange(0,vmax+1e-9,step)
    if ticks[-1]<vmax:ticks=np.append(ticks,vmax)
    cb=fig.colorbar(image,cax=cax,orientation="horizontal",ticks=ticks)
    cb.set_label("Relative contribution (%)",labelpad=5);cb.outline.set_linewidth(0.5)
    name=FIGURE_NAMES[stage]

    fig.savefig(FIGURES / f'{name}.png', dpi=DPI, facecolor='white')
    with Image.open(FIGURES / f'{name}.png') as image:
        image.convert('RGB').save(FIGURES / f'{name}.tiff', compression='tiff_lzw', dpi=(DPI, DPI))
    plt.close(fig)
